# 第 5 章 · 假设检验

**假设检验**判断"样本里的差异，是真实存在的还是碰巧抽到的"。
本章覆盖 t 检验、卡方检验和方差分析。

## 0. 准备环境

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 matplotlib 正确显示中文与负号
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)   # 固定随机种子，保证每次运行结果一致
print("环境就绪：numpy", np.__version__, "| pandas", pd.__version__)
import scipy.stats as st

## 1. 假设检验的基本框架

1. 建立 **原假设 H₀**（通常"没有差异/没有效应"）与 **备择假设 H₁**。
2. 选择显著性水平 α（常取 0.05）。
3. 计算检验统计量与 **p 值**。
4. 判断：p < α → 拒绝 H₀；否则不拒绝 H₀。

**注意**：p 值不是"原假设为真的概率"，"不拒绝"也不等于"证明 H₀ 正确"。

## 2. 单样本 t 检验

问题：某班数学平均分是否显著**高于** 70 分？

In [ ]:
df = pd.read_csv("data/students.csv")
scores = df["数学"]

t_stat, p_value = st.ttest_1samp(scores, popmean=70, alternative="greater")
print("样本均值 = %.2f" % scores.mean())
print("t 统计量 = %.4f,  p 值 = %.4f" % (t_stat, p_value))
print("结论：", "拒绝 H0，平均分显著高于 70" if p_value < 0.05 else "不拒绝 H0")

## 3. 独立双样本 t 检验

问题：男生和女生的数学成绩是否有显著差异？

In [ ]:
male = df.loc[df["性别"] == "男", "数学"]
female = df.loc[df["性别"] == "女", "数学"]

t_stat, p_value = st.ttest_ind(male, female, equal_var=False)   # Welch t 检验
print("男 n=%d 均值=%.2f | 女 n=%d 均值=%.2f" % (len(male), male.mean(), len(female), female.mean()))
print("t = %.4f, p = %.4f" % (t_stat, p_value))
print("结论：", "存在显著差异" if p_value < 0.05 else "无显著差异")

## 4. 配对样本 t 检验

问题：同一批学生，数学和语文成绩是否有系统差异？（配对设计）

In [ ]:
t_stat, p_value = st.ttest_rel(df["数学"], df["语文"])
print("数学均值 = %.2f | 语文均值 = %.2f" % (df["数学"].mean(), df["语文"].mean()))
print("t = %.4f, p = %.4f" % (t_stat, p_value))
print("结论：", "两科成绩存在系统差异" if p_value < 0.05 else "两科成绩无系统差异")

## 5. 卡方独立性检验

问题：**性别**与**是否偏好某科目**有关联吗？用列联表检验两个分类变量是否独立。

In [ ]:
# 构造列联表：性别 × 班级
table = pd.crosstab(df["性别"], df["班级"])
print(table)

chi2, p_value, dof, expected = st.chi2_contingency(table)
print("\n卡方统计量 = %.4f, 自由度 = %d, p 值 = %.4f" % (chi2, dof, p_value))
print("结论：", "两个变量不独立（有关联）" if p_value < 0.05 else "未发现关联")

## 6. 单因素方差分析（One-way ANOVA）

问题：三个班级的数学平均分是否**至少有一组**不同？

In [ ]:
groups = [g["数学"].values for _, g in df.groupby("班级")]
f_stat, p_value = st.f_oneway(*groups)

for name, g in df.groupby("班级"):
    print("%s: n=%d, 均值=%.2f" % (name, len(g), g["数学"].mean()))
print("\nF 统计量 = %.4f, p 值 = %.4f" % (f_stat, p_value))
print("结论：", "至少有一个班级均值不同" if p_value < 0.05 else "各班级均值无显著差异")

## 小结

| 场景 | 方法 |
|---|---|
| 单组 vs 已知均值 | 单样本 t 检验 |
| 两组独立 | 独立双样本 t 检验 |
| 两组配对 | 配对 t 检验 |
| 两个分类变量 | 卡方检验 |
| 三组及以上 | 方差分析 ANOVA |

**核心口诀**：p < α 拒绝原假设；永远结合效应量和实际问题来解释。